# 04 - GAT Training

Goal: train a Graph Attention Network for binary edge classification.

The graph has IP nodes and traffic-flow edges. The model predicts whether each edge is benign or malicious.

Validation is used to select the best epoch and classification threshold. The test split is used only for final reporting.

In [ ]:
from argparse import Namespace
from pathlib import Path
import json
import sys

import torch

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

from src.train_gat import train  # noqa: E402

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

## Training configuration

This configuration aims for high scores without reporting a suspicious perfect result. Early stopping keeps the best validation model.

In [ ]:
args = Namespace(
    arrays_path=PROJECT_ROOT / "data" / "processed" / "graph_arrays.npz",
    model_dir=PROJECT_ROOT / "models",
    results_dir=PROJECT_ROOT / "results",
    epochs=50,
    hidden_channels=64,
    heads=4,
    dropout=0.2,
    lr=0.005,
    weight_decay=0.0005,
    random_state=42,
    cpu=False,
    early_stopping=True,
    patience=8,
    min_delta=0.0005,
    selection_metric="val_f1",
    message_passing_edges="all",
)

args

## Train and evaluate

The script reports Accuracy, Precision, Recall, F1-score, ROC-AUC, PR-AUC, and confusion matrices.

In [ ]:
results = train(args)

results["final_metrics"]

## Saved outputs and generalization check

In [ ]:
metrics_path = PROJECT_ROOT / "results" / "gat_metrics.json"
model_path = PROJECT_ROOT / "models" / "gat_edge_classifier.pt"

print("Metrics saved:", metrics_path.exists(), metrics_path)
print("Model saved:", model_path.exists(), model_path)

saved = json.loads(metrics_path.read_text(encoding="utf-8"))
print("Best epoch:", saved["best_epoch"])
print("Selected threshold:", saved["threshold_selection"]["threshold"])
print("Train-test gap:", saved["generalization_gap_train_minus_test"])
saved["final_metrics"]["test"]